# 🏙️ Smart City Autonomous Driving — TrafficFSM + NavigationFSM + Road AI

This production notebook implements the **Smart City Autonomous System** with:

| Module | Class | Description |
| :--- | :--- | :--- |
| **Traffic Sign AI** | `YOLOProcessor` + `ONNXEngine` | Real-time traffic sign & light detection with custom NumPy NMS |
| **Road Status AI** | `RoadProcessor` + `ONNXEngine` | Path safety classification (`FREE` vs `BLOCKED`) |
| **Traffic FSM** | `TrafficFSM` | Bounding box spatial filtering (`ROI 5%-95%`), sign priority resolution, and direction list generation |
| **Navigation FSM** | `NavigationFSM` | 2-Phase obstacle escape state machine (`DRIVE` -> `REVERSE_TURNING` -> `PAUSE` -> `CHECK_FORWARD`) |
| **Racecar HAL** | `RacecarController` | Hardware abstraction layer for motor steering & throttle control with safe shutdown handlers |


In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys

# Add project root and jetracer directories to sys.path
current_dir = os.getcwd()
sys.path.insert(0, current_dir)
sys.path.insert(0, os.path.join(current_dir, 'jetracer'))

parent_dir = os.path.dirname(current_dir)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
grandparent_dir = os.path.dirname(parent_dir)
if grandparent_dir not in sys.path:
    sys.path.insert(0, grandparent_dir)


In [ ]:
import sys
import os
import cv2
import numpy as np
import onnxruntime as ort
import rospy
from sensor_msgs.msg import Image

print(f"[+] ONNX Runtime Version: {ort.__version__}")

# Check execution providers (GPU TensorRT / CUDA priority)
available_providers = ort.get_available_providers()
print(f"[+] Available Providers: {available_providers}")
if 'CUDAExecutionProvider' in available_providers or 'TensorrtExecutionProvider' in available_providers:
    print("[✓] GPU Acceleration Enabled (CUDA / TensorRT)!")
else:
    print("[!] Running on CPU provider fallback.")


In [ ]:
import gc
import time
import cv2
import numpy as np
import rospy
from IPython.display import Image, clear_output, display

from smart_city.camera_steam import CameraStream 
from smart_city.processor import YOLOProcessor, RoadProcessor
from smart_city.traffic_fsm import TrafficFSM
from smart_city.onnx_engine import ONNXEngine
from smart_city.controller import RacecarController
from smart_city.navigation_fsm import NavigationFSM

# Initialize ONNX Engines
traffic_engine = ONNXEngine("../models/best.onnx")
road_engine    = ONNXEngine("../models/best_model_mobilenet.onnx")

# Initialize Processors
traffic_processor = YOLOProcessor(img_size=640, conf_thresh=0.50)
road_processor    = RoadProcessor(img_size=(224, 224), threshold=0.50)

# Initialize Camera Stream
cam = CameraStream(
    topic_name='/csi_cam_0/image_raw',
    width=640,
    height=480,
    record_video=False
)

print("✅ AI Engines, Processors & Camera Stream Active!")


In [ ]:
from smart_city.controller import RacecarController

car = RacecarController(base_throttle=0.15)


In [ ]:
car.stop()
print("🛑 Car Halted.")


## 🚗 Main Production Cell: Autonomous Driving Loop + NavigationFSM

Click `🚀 START` to activate driving, `🛑 STOP` to halt.

In [ ]:
import gc, time, cv2, rospy, threading, ipywidgets as w, logging
from IPython.display import display

# --- 1. CONFIGURATION & STATE MACHINES ---
fsm = TrafficFSM(
    conf_threshold=0.5, 
    min_consecutive_frames=5, 
    min_bbox_area_sign=2000, 
    min_bbox_area_traffic_light=1000, 
    roi_x_min=0.05, 
    roi_x_max=0.95
)

nav_fsm = NavigationFSM(
    confirm_frames=3,
    block_threshold=0.8,
    stop_timeout=30.0,
    max_reverse_cycles=5
)

is_running = False
is_loop_active = False
last_logged_state = None

# --- 2. INTERACTIVE UI CONTROLS ---
btn_start = w.Button(description="🚀 START", button_style='success', layout=w.Layout(width='160px'))
btn_stop  = w.Button(description="🛑 STOP",  button_style='danger',  layout=w.Layout(width='160px'))
image_widget = w.Image(format='jpeg', width=640)

def on_start_clicked(b):
    global is_running
    is_running = True
    nav_fsm.reset()
    rospy.loginfo("▶️ START clicked: Autonomous Driving Active!")

def on_stop_clicked(b):
    global is_running
    is_running = False
    car.stop()
    rospy.loginfo("⏸️ STOP clicked: Racecar Halted!")

btn_start.on_click(on_start_clicked)
btn_stop.on_click(on_stop_clicked)

display(w.HBox([btn_start, btn_stop]))
display(image_widget)

def resolve_intended_direction(allowed_dirs):
    """Resolves direction list from TrafficFSM into single target direction for NavigationFSM."""
    if not allowed_dirs or "STOP" in allowed_dirs:
        return "STOP"
    if any(d in allowed_dirs for d in ["FORWARD", "STRAIGHT"]):
        return "FORWARD"
    if any(d in allowed_dirs for d in ["RIGHT", "TURN_RIGHT"]):
        return "RIGHT"
    if any(d in allowed_dirs for d in ["LEFT", "TURN_LEFT"]):
        return "LEFT"
    return "FORWARD"

# --- 3. MAIN CONTROL LOOP (THREADED WORKER) ---
def camera_control_loop():
    global is_running, is_loop_active, last_logged_state
    
    rate, frame_count = rospy.Rate(20), 0
    prob_blocked = 0.0

    try:
        while is_loop_active and not rospy.is_shutdown():
            frame = cam.get_frame()
            if frame is None:
                car.stop()
                rate.sleep()
                continue

            frame_count += 1
            now = rospy.get_time()

            # --- A. TRAFFIC AI INFERENCE ---
            t_input, orig_h, orig_w = traffic_processor.preprocess(frame)
            detections = traffic_processor.postprocess(traffic_engine.infer(t_input), orig_h, orig_w)
            allowed_dirs = fsm.update(detections, img_w=orig_w, img_h=orig_h)

            # --- B. ROAD BLOCK SAFETY INFERENCE ---
            intended_dir = resolve_intended_direction(allowed_dirs)

            r_input = road_processor.preprocess(frame)
            raw_road_out = road_engine.infer(r_input)
            road_res = road_processor.postprocess(raw_road_out)
            prob_blocked = road_res['blocked_probability']

            # --- C. NAVIGATION FSM & MOTOR EXECUTION ---
            if is_running:
                maneuver_state, blocked_cnt = nav_fsm.update(intended_dir, prob_blocked, car, now)
            else:
                car.stop()
                maneuver_state, blocked_cnt = 'PAUSE', 0

            # --- D. LOGGING & CONSOLE FEEDBACK ---
            current_state = (is_running, maneuver_state, blocked_cnt, tuple(allowed_dirs), car.car.steering, car.car.throttle)
            if current_state != last_logged_state:
                status_tag = "RUNNING" if is_running else "PAUSED"
                rospy.loginfo(f"[{status_tag}] NavState: {maneuver_state} | BlkCnt: {blocked_cnt} | Dir: {intended_dir} | Steer: {car.car.steering:.2f} | Thr: {car.car.throttle:.2f}")
                last_logged_state = current_state

            # --- E. VISUALIZATION OVERLAY ---
            if frame_count % 2 == 0:
                vis_frame = frame.copy()
                for d in detections:
                    if d.get("confidence", 0) >= fsm.conf_threshold:
                        b = d.get("bbox") or d.get("box")
                        x1, y1, x2, y2 = (int(b[0]), int(b[1]), int(b[0]+b[2]), int(b[1]+b[3])) if "box" in d else (int(b[0]), int(b[1]), int(b[2]), int(b[3]))
                        cv2.rectangle(vis_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                
                color = (0, 255, 0) if is_running else (0, 0, 255)
                cv2.putText(vis_frame, f"Mode: {maneuver_state} | BlkCnt: {blocked_cnt}", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
                cv2.putText(vis_frame, f"Intended Dir: {intended_dir} | Steer: {car.car.steering:+.2f} | Thr: {car.car.throttle:.2f}", (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 0), 2)

                image_widget.value = cv2.imencode('.jpg', vis_frame)[1].tobytes()

            if frame_count % 100 == 0: 
                gc.collect()

            rate.sleep()

    finally:
        car.stop()
        rospy.loginfo('🛑 Car Stopped Safely.')

# --- 4. START THREAD ---
is_loop_active = True
thread = threading.Thread(target=camera_control_loop, daemon=True)
thread.start()


## 🔍 Debug Cell: TrafficFSM BBox & ROI Visualization

Visualizes ROI bounds and valid/invalid detections in real-time.

In [ ]:
import time
import cv2
import gc
import rospy
import ipywidgets as w
from IPython.display import display

# --- 1. CONFIGURATION & TRAFFIC FSM INITIALIZATION ---
DISPLAY_UI, SKIP_FRAMES, TARGET_FPS = True, 1, 20

fsm = TrafficFSM(
    conf_threshold=0.5, 
    min_consecutive_frames=5, 
    min_bbox_area_sign=2000, 
    min_bbox_area_traffic_light=1000, 
    roi_x_min=0.05, 
    roi_x_max=0.95
)

image_widget = w.Image(format='jpeg', width=640)
if DISPLAY_UI:
    display(image_widget)

rate = rospy.Rate(TARGET_FPS)
frame_count = 0

rospy.loginfo('🚀 Starting Real-time Debug Stream - TrafficFSM Test...')

try:
    while not rospy.is_shutdown():
        frame = cam.get_frame()
        if frame is None:
            rospy.logwarn_throttle(2.0, '⏳ Waiting for camera frame...')
            rate.sleep()
            continue

        start_time = rospy.get_time()
        frame_count += 1

        # --- A. AI INFERENCE ---
        t_input, orig_h, orig_w = traffic_processor.preprocess(frame)
        detections = traffic_processor.postprocess(
            traffic_engine.infer(t_input), orig_h, orig_w
        )

        # --- B. UPDATE TRAFFIC FSM ---
        allowed_dirs = fsm.update(detections, img_w=orig_w, img_h=orig_h)

        # --- C. VISUALIZATION OVERLAY ---
        latency_ms = (rospy.get_time() - start_time) * 1000
        fps_real = 1000 / max(latency_ms, 1)

        if DISPLAY_UI and (frame_count % SKIP_FRAMES == 0):
            vis_frame = frame.copy()

            # 1. Draw ROI X bounds (2 vertical yellow lines)
            rx_min, rx_max = int(orig_w * fsm.roi_x_min), int(orig_w * fsm.roi_x_max)
            cv2.line(vis_frame, (rx_min, 0), (rx_min, orig_h), (255, 255, 0), 2)
            cv2.line(vis_frame, (rx_max, 0), (rx_max, orig_h), (255, 255, 0), 2)

            # 2. Bounding Box & Valid Filtering Visualization
            for d in detections:
                label, conf = d.get("class_name", ""), d.get("confidence", 0)
                b = d.get("bbox") or d.get("box")
                if b is None or conf < fsm.conf_threshold: 
                    continue

                if "box" in d:
                    x1, y1, x2, y2 = int(b[0]), int(b[1]), int(b[0] + b[2]), int(b[1] + b[3])
                else:
                    x1, y1, x2, y2 = int(b[0]), int(b[1]), int(b[2]), int(b[3])

                area, center_x = fsm._parse_box(d, orig_w)
                is_valid = fsm._is_valid_bbox(d, orig_w, orig_h) and (label in fsm.priority)

                color = (0, 255, 0) if is_valid else (0, 0, 255)
                cv2.rectangle(vis_frame, (x1, y1), (x2, y2), color, 2)
                
                label_text = f"{label} {conf:.2f} ({int(area)}px)"
                cv2.putText(vis_frame, label_text, (x1, max(20, y1 - 5)), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)

            # 3. Draw FSM Info Overlay
            cv2.putText(vis_frame, f"FPS: {fps_real:.1f} | Latency: {latency_ms:.1f}ms", 
                        (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            
            status_color = (0, 0, 255) if allowed_dirs == ["STOP"] else (0, 255, 0)
            cv2.putText(vis_frame, f"Allowed Directions: {allowed_dirs}", 
                        (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, status_color, 2)

            image_widget.value = cv2.imencode('.jpg', vis_frame)[1].tobytes()

        if frame_count % 100 == 0:
            gc.collect()

        rate.sleep()

except KeyboardInterrupt:
    rospy.loginfo('🛑 Stopped FSM debug stream by KeyboardInterrupt.')
finally:
    rospy.loginfo('🛑 Test finished.')


## 🛤️ Test Cell: Lane Following (BEV + Sliding Window)

2x2 grid stream showing raw frame, BEV perspective, binary mask, and sliding window center tracking.

In [ ]:
import cv2
import time
import numpy as np
import rospy
from IPython.display import display, Image, clear_output
from smart_city.lane_follower import LaneFollower

# Initialize Lane Follower
lane_follower = LaneFollower(img_w=640, img_h=480)

display_handle = display(None, display_id=True)
print("Running Realtime Lane Following Test. Click Stop on toolbar to exit.")

try:
    fps_start_time = time.time()
    frame_count = 0
    fps = 0.0

    while not rospy.is_shutdown():
        frame = cam.get_frame()
        if frame is None:
            time.sleep(0.005)
            continue

        frame_count += 1
        if frame_count >= 10:
            elapsed = time.time() - fps_start_time
            fps = frame_count / elapsed if elapsed > 0 else 0.0
            fps_start_time = time.time()
            frame_count = 0

        # Process BEV and Sliding Window
        bev_color = lane_follower.get_bev(frame)
        binary_lane = lane_follower.get_binary_lane(bev_color)
        steering, speed, vis_bev = lane_follower.process_sliding_window(binary_lane, allowed_dirs=["FORWARD"])

        car.set_steering(steering)
        car.set_throttle(0.1)

        # 2x2 Grid Visualization
        binary_3ch = cv2.cvtColor(binary_lane, cv2.COLOR_GRAY2BGR)

        h_v, w_v = 240, 320
        img1 = cv2.resize(frame, (w_v, h_v))
        img2 = cv2.resize(bev_color, (w_v, h_v))
        img3 = cv2.resize(binary_3ch, (w_v, h_v))
        img4 = cv2.resize(vis_bev, (w_v, h_v))

        cv2.putText(img1, f"Steer: {steering:.2f} | Speed: {speed:.2f}", (10, 25), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        cv2.putText(img1, f"FPS: {fps:.1f}", (10, 50), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)

        top_row = np.hstack((img1, img2))
        bot_row = np.hstack((img3, img4))
        grid_vis = np.vstack((top_row, bot_row))

        if frame_count % 5 == 0:
            _, jpeg = cv2.imencode('.jpg', grid_vis, [int(cv2.IMWRITE_JPEG_QUALITY), 50])
            display_handle.update(Image(data=jpeg.tobytes()))

except (KeyboardInterrupt, SystemExit):
    print("\nReceived interrupt signal.")
finally:
    car.stop()
    print("Lane Following Test Stopped. Motor Halted.")


In [ ]:
is_running = False
is_loop_active = False
car.stop()
print("🛑 SmartCity System Shutdown Cleanly.")
